# Práctica 06: Clasificación de los Pacientes con Riesgo de  Infarto Cardio Vascular en el Estado de Puebla

**Proyecto:** Clasificación de los Pacientes con Riesgo de Infarto Cardio Vascular en el Estado de Puebla

Este notebook desarrolla la Práctica 02: se parte del dataset clínico simulado (5,000 pacientes) al que se le agregaron coordenadas GPS (latitud/longitud) generadas dentro de los polígonos reales de cada municipio de Puebla. A partir de ahí se realiza:

1. Revisión y validación del dataset (estructura, nulos, geoposicionamiento)
2. Carga de polígonos municipales con GeoPandas
3. Unión espacial (spatial join) para obtener `municipio_objetivo`
4. Análisis y visualización de la distribución geográfica de pacientes
5. Mapas interactivos: puntos, heatmap y coroplético (Folium)
6. Preparación del dataset de modelado (**solo latitud/longitud como variables predictoras**)
7. Entrenamiento de modelos supervisados de clasificación (KNN, Árbol de Decisión / Random Forest, SVM)
8. Evaluación, matriz de confusión y análisis de errores
9. Predicción sobre nuevas coordenadas y comparación contra la asignación espacial
10. Conclusiones

> **Nota sobre el trabajo en equipo:** en la práctica original, cada integrante genera su propio dataset de 5,000 pacientes y el equipo los fusiona en un dataset grupal de 20,000–30,000 registros. Este notebook está construido para trabajar con **un** dataset individual; si tu equipo ya fusionó los datasets, simplemente reemplaza el archivo cargado en la sección 1 por el CSV grupal (debe conservar las mismas columnas, incluyendo `Latitud` y `Longitud`) y el resto del notebook funcionará igual.

## 1. Carga del dataset y verificación de estructura

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

df = pd.read_csv('dataset_riesgo_infarto_puebla_5000.csv', encoding='utf-8-sig')
print(f"Dimensiones: {df.shape[0]} pacientes x {df.shape[1]} columnas")
df.head()


Dimensiones: 5000 pacientes x 27 columnas


,ID_Paciente,Nombre,Apellido,Sexo,Edad,Municipio,Latitud,Longitud,Peso_kg,Estatura_m,IMC,Presion_Sistolica,Presion_Diastolica,Frecuencia_Cardiaca,Temperatura,Saturacion_O2,Glucosa,Colesterol,Trigliceridos,Fuma,Alcohol,Actividad_Fisica,Diabetes,Hipertension,Obesidad,Diagnostico,Riesgo
0,1,Juan,Pérez,Masculino,21,Tehuacán,18.511587,-97.267586,76.3,1.75,24.91,135,75,68,36.5,92,127,165,258,Sí,Sí,Media,Sí,No,No,Paciente sano,Bajo
1,2,Carlos,Hernández,Masculino,87,San Pedro Cholula,19.120649,-98.305600,96.3,1.77,30.74,153,74,103,36.7,93,184,280,141,Sí,Sí,Media,Sí,Sí,Sí,Obesidad,Alto
2,3,José,Sánchez,Femenino,30,San Andrés Cholula,19.051854,-98.207348,65.2,1.58,26.12,145,82,108,37.4,97,137,141,256,No,Sí,Alta,Sí,Sí,No,Gastritis,Medio
3,4,Laura,González,Masculino,26,Atlixco,18.969397,-98.467407,87.2,1.83,26.04,105,74,78,36.8,96,144,150,288,Sí,Sí,Baja,Sí,No,No,Paciente sano,Bajo
4,5,Juan,López,Masculino,39,Puebla,18.939616,-98.163867,84.5,1.69,29.59,168,75,106,36.7,96,111,248,167,No,Sí,Media,No,Sí,No,Paciente sano,Bajo


In [2]:
# Confirmar columnas de geoposicionamiento
assert {'Latitud', 'Longitud', 'Municipio'}.issubset(df.columns), "Faltan columnas de geoposicionamiento"
print("Columnas de geoposicionamiento presentes: Latitud, Longitud, Municipio")
df[['Latitud', 'Longitud']].describe()


Columnas de geoposicionamiento presentes: Latitud, Longitud, Municipio


,Latitud,Longitud
count,5000.000000,5000.000000
mean,19.271120,-97.964230
std,0.515162,0.353980
min,18.401012,-98.468618
25%,18.992329,-98.207852
50%,19.098043,-98.041778
75%,19.876689,-97.910925
max,20.286648,-97.190656


### 1.1 Validar número total de registros (dataset grupal)

In [3]:
# Si trabajas con el dataset grupal fusionado, aquí deberías ver ~ (num_integrantes x 5000) registros
print(f"Total de registros: {len(df)}")
print(f"Número de integrantes esperado (aprox.): {len(df) / 5000:.1f}")


Total de registros: 5000
Número de integrantes esperado (aprox.): 1.0


### 1.2 Duplicados y valores nulos

In [4]:
print("Registros duplicados (todas las columnas):", df.duplicated().sum())
print("IDs de paciente duplicados:", df['ID_Paciente'].duplicated().sum())
print("\nNulos en columnas de geoposicionamiento:")
print(df[['Latitud', 'Longitud']].isnull().sum())

# Eliminar duplicados si los hubiera (por seguridad, ej. tras fusionar equipos)
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nRegistros eliminados por duplicado: {antes - len(df)}")


Registros duplicados (todas las columnas): 0
IDs de paciente duplicados: 0

Nulos en columnas de geoposicionamiento:
Latitud     0
Longitud    0
dtype: int64

Registros eliminados por duplicado: 0


### 1.3 Validar rangos de coordenadas (deben caer dentro del territorio de Puebla)

In [5]:
# Bounding box aproximado del estado de Puebla
LAT_MIN, LAT_MAX = 17.8, 20.9
LON_MIN, LON_MAX = -99.1, -96.5

fuera_de_rango = df[~df['Latitud'].between(LAT_MIN, LAT_MAX) | ~df['Longitud'].between(LON_MIN, LON_MAX)]
print(f"Registros con coordenadas fuera del rango esperado de Puebla: {len(fuera_de_rango)}")

if len(fuera_de_rango) > 0:
    df = df.drop(fuera_de_rango.index).reset_index(drop=True)
    print("Registros fuera de rango eliminados.")


Registros con coordenadas fuera del rango esperado de Puebla: 0


In [6]:
# Copia completa (con todas las etiquetas originales) para consulta y validación,
# NO se usará para entrenar el modelo.
df_referencia = df.copy()
print("Copia de referencia guardada en 'df_referencia' (incluye Municipio, Diagnostico, Riesgo, etc.)")


Copia de referencia guardada en 'df_referencia' (incluye Municipio, Diagnostico, Riesgo, etc.)


## 2. Carga de polígonos municipales con GeoPandas

Se utilizan los límites municipales oficiales del estado de Puebla (213 municipios, fuente: CONABIO / repositorio público `angelnmara/geojson`).

In [7]:
municipios_gdf = gpd.read_file('puebla_municipios.geojson')[['NAME_2', 'geometry']]
municipios_gdf = municipios_gdf.rename(columns={'NAME_2': 'municipio_poligono'})
print(f"Polígonos municipales cargados: {len(municipios_gdf)}")
print(f"Sistema de referencia (CRS): {municipios_gdf.crs}")
municipios_gdf.head()


Polígonos municipales cargados: 213
Sistema de referencia (CRS): EPSG:4326


,municipio_poligono,geometry
0,Acajete,"POLYGON ((-97.75841 19.14529, -97.75938 19.142..."
1,Acateno,"POLYGON ((-97.15316 20.0896, -97.15682 20.0905..."
2,Acatlán,"POLYGON ((-97.97835 18.16613, -97.97736 18.169..."
3,Acatzingo,"POLYGON ((-97.72066 19.02214, -97.72065 19.025..."
4,Acteopan,"POLYGON ((-98.53517 18.75965, -98.53642 18.756..."


In [8]:
# El CRS de los polígonos es EPSG:4326 (WGS84, grados lat/lon), el mismo sistema
# en que están capturadas las coordenadas de los pacientes. No se requiere reproyección.
print("CRS polígonos municipales:", municipios_gdf.crs)


CRS polígonos municipales: EPSG:4326


## 3. Conversión del dataset de pacientes a GeoDataFrame

In [9]:
geometry_pacientes = [Point(xy) for xy in zip(df['Longitud'], df['Latitud'])]
pacientes_gdf = gpd.GeoDataFrame(df.copy(), geometry=geometry_pacientes, crs='EPSG:4326')
pacientes_gdf.head()


,ID_Paciente,Nombre,Apellido,Sexo,Edad,Municipio,Latitud,Longitud,Peso_kg,Estatura_m,IMC,Presion_Sistolica,Presion_Diastolica,Frecuencia_Cardiaca,Temperatura,Saturacion_O2,Glucosa,Colesterol,Trigliceridos,Fuma,Alcohol,Actividad_Fisica,Diabetes,Hipertension,Obesidad,Diagnostico,Riesgo,geometry
0,1,Juan,Pérez,Masculino,21,Tehuacán,18.511587,-97.267586,76.3,1.75,24.91,135,75,68,36.5,92,127,165,258,Sí,Sí,Media,Sí,No,No,Paciente sano,Bajo,POINT (-97.26759 18.51159)
1,2,Carlos,Hernández,Masculino,87,San Pedro Cholula,19.120649,-98.305600,96.3,1.77,30.74,153,74,103,36.7,93,184,280,141,Sí,Sí,Media,Sí,Sí,Sí,Obesidad,Alto,POINT (-98.3056 19.12065)
2,3,José,Sánchez,Femenino,30,San Andrés Cholula,19.051854,-98.207348,65.2,1.58,26.12,145,82,108,37.4,97,137,141,256,No,Sí,Alta,Sí,Sí,No,Gastritis,Medio,POINT (-98.20735 19.05185)
3,4,Laura,González,Masculino,26,Atlixco,18.969397,-98.467407,87.2,1.83,26.04,105,74,78,36.8,96,144,150,288,Sí,Sí,Baja,Sí,No,No,Paciente sano,Bajo,POINT (-98.46741 18.9694)
4,5,Juan,López,Masculino,39,Puebla,18.939616,-98.163867,84.5,1.69,29.59,168,75,106,36.7,96,111,248,167,No,Sí,Media,No,Sí,No,Paciente sano,Bajo,POINT (-98.16387 18.93962)


### 3.1 Visualización inicial de los puntos de pacientes